# 05 — Combine Candidates, Labels, Recall & Features

**Purpose:** turn candidate sources into the table the ranker trains/scores on.

Fixes vs the old notebook:
- No more `MemoryError` (IDs are int now, set in notebook 01).
- The feature tables are **actually merged** onto the candidates (the old version computed
  them and then never joined them, and referenced an undefined `ranking_df`).
- Adds a **candidate-recall** diagnostic (the retrieval ceiling).
- Down-samples negatives for the TRAIN frame; keeps the VAL frame intact for scoring.

Built for both weeks: `ranking_train.parquet` (labels + down-sampled negs) and
`ranking_val.parquet` (full candidates for the holdout score).

In [1]:
from recsys_utils import *
import numpy as np, pandas as pd
mlflow = setup_mlflow()

transactions = load_parquet(PROCESSED_DIR / 'transactions_clean.parquet')
articles     = load_parquet(PROCESSED_DIR / 'articles_clean.parquet')
cfg          = load_json(PROCESSED_DIR / 'split_config.json')
train_truth  = load_parquet(PROCESSED_DIR / 'train_truth.parquet')
val_truth    = load_parquet(PROCESSED_DIR / 'val_truth.parquet')

In [2]:
TREND_ADD       = 30    # top trending items added to every week-user (cold-start coverage)
NEG_PER_USER    = 100   # cap on sampled negatives per user in the TRAIN frame
META_COLS = ['product_type_name', 'product_group_name', 'colour_group_name',
             'department_name', 'index_group_name', 'garment_group_name']

## Combine + de-duplicate candidate sources

In [3]:
def combine_candidates(tag, users):
    rep = load_parquet(CANDIDATE_DIR / f'repurchase_{tag}.parquet')[['customer_id', 'article_id', 'source']]
    als = load_parquet(CANDIDATE_DIR / f'als_candidates_{tag}.parquet')
    trd = load_parquet(CANDIDATE_DIR / f'trending_{tag}.parquet')

    base = pd.concat([rep, als[['customer_id', 'article_id', 'als_score', 'als_rank', 'source']]],
                     ignore_index=True)
    base = base.drop_duplicates(['customer_id', 'article_id'])   # repurchase wins ties over als

    top_trend = trd.head(TREND_ADD)[['article_id', 'trending_rank']]
    users_df  = pd.DataFrame({'customer_id': np.asarray(users, dtype='int32')})
    trend_exp = users_df.merge(top_trend, how='cross')
    trend_exp['source'] = 'trending'

    cands = pd.concat([base, trend_exp], ignore_index=True)
    cands = cands.drop_duplicates(['customer_id', 'article_id']).reset_index(drop=True)
    return cands

## Labels (purchased in the next 7 days = 1)

In [4]:
def attach_labels(cands, truth):
    pos = (truth.explode('actual').rename(columns={'actual': 'article_id'})[['customer_id', 'article_id']])
    pos['article_id'] = pos['article_id'].astype('int32')
    pos['label'] = np.int8(1)
    out = cands.merge(pos, on=['customer_id', 'article_id'], how='left')
    out['label'] = out['label'].fillna(0).astype('int8')
    return out

## Feature engineering (user / item / user x item / source)

All features use only data strictly before the week's start (`basis_end`).

In [5]:
def build_features(cands, basis_end):
    hist = transactions[transactions['t_dat'] < basis_end]

    # ---- user features ----
    u = (hist.groupby('customer_id')
             .agg(user_n_tx=('article_id', 'count'),
                  user_avg_spend=('price', 'mean'),
                  user_n_distinct=('article_id', 'nunique'),
                  user_last=('t_dat', 'max')).reset_index())
    u['user_days_since_last'] = (basis_end - u['user_last']).dt.days.astype('int32')
    u = u.drop(columns='user_last')
    for d in (7, 30, 90):
        c = (hist[hist['t_dat'] >= basis_end - pd.Timedelta(days=d)]
                 .groupby('customer_id')['article_id'].count().rename(f'user_cnt_{d}d'))
        u = u.merge(c, on='customer_id', how='left')

    # ---- item features ----
    i = (hist.groupby('article_id')
             .agg(item_n_tx=('customer_id', 'count'),
                  item_avg_price=('price', 'mean'),
                  item_first=('t_dat', 'min')).reset_index())
    i['item_days_since_first'] = (basis_end - i['item_first']).dt.days.astype('int32')
    i = i.drop(columns='item_first')
    s7 = (hist[hist['t_dat'] >= basis_end - pd.Timedelta(days=7)]
             .groupby('article_id').size().rename('item_sales_7d'))
    sp = (hist[(hist['t_dat'] >= basis_end - pd.Timedelta(days=14)) &
               (hist['t_dat'] <  basis_end - pd.Timedelta(days=7))]
             .groupby('article_id').size().rename('item_sales_prior7d'))
    i = i.merge(s7, on='article_id', how='left').merge(sp, on='article_id', how='left')
    i[['item_sales_7d', 'item_sales_prior7d']] = i[['item_sales_7d', 'item_sales_prior7d']].fillna(0)
    i['item_velocity']     = i['item_sales_7d'] - i['item_sales_prior7d']
    i['item_price_pct']    = i['item_avg_price'].rank(pct=True)

    # ---- user x item: did the user buy this exact article before? ----
    ub = hist[['customer_id', 'article_id']].drop_duplicates()
    ub['ui_bought_before'] = np.int8(1)

    # ---- merge everything onto candidates ----
    df = (cands.merge(u, on='customer_id', how='left')
                .merge(i, on='article_id', how='left')
                .merge(ub, on=['customer_id', 'article_id'], how='left')
                .merge(articles[['article_id'] + META_COLS], on='article_id', how='left'))

    df['ui_price_ratio'] = df['item_avg_price'] / (df['user_avg_spend'] + 1e-6)

    # candidate-source flags + scores
    df['is_repurchase'] = (df['source'] == 'repurchase').astype('int8')
    df['is_als']        = (df['source'] == 'als').astype('int8')
    df['is_trending']   = (df['source'] == 'trending').astype('int8')
    df['als_score']    = df.get('als_score', pd.Series(0.0, index=df.index)).fillna(0.0).astype('float32')
    df['als_rank']     = df.get('als_rank', pd.Series(999, index=df.index)).fillna(999).astype('int16')
    df['trending_rank']= df.get('trending_rank', pd.Series(999, index=df.index)).fillna(999).astype('int16')
    df['ui_bought_before'] = df['ui_bought_before'].fillna(0).astype('int8')

    # numeric NaNs -> 0 ; categoricals -> 'Unknown' (LightGBM will read these as category dtype)
    num_cols = df.select_dtypes(include=['float64', 'float32', 'int64', 'int32', 'int16']).columns
    df[num_cols] = df[num_cols].fillna(0)
    for c in META_COLS:
        df[c] = df[c].fillna('Unknown').astype('category')
    return df

## Candidate recall (retrieval ceiling) — measured on the VAL week

In [6]:
val_cands = combine_candidates('val', val_truth['customer_id'].values)
cand_map  = val_cands.groupby('customer_id')['article_id'].apply(list).to_dict()
cand_lists = [cand_map.get(u, []) for u in val_truth['customer_id']]
rec = candidate_recall(val_truth['actual'].tolist(), cand_lists)
avg_cands = np.mean([len(c) for c in cand_lists])
print(f'Candidate recall on VAL week : {rec:.4f}')
print(f'Avg candidates per val user  : {avg_cands:.1f}')
print('If recall is low, fix retrieval (more ALS top-N / trending) BEFORE the ranker.')

Candidate recall on VAL week : 0.1046
Avg candidates per val user  : 134.5
If recall is low, fix retrieval (more ALS top-N / trending) BEFORE the ranker.


## Build the TRAIN frame (labels + down-sampled negatives) and the VAL frame

In [7]:
def downsample_negatives(df, neg_per_user=NEG_PER_USER, seed=RANDOM_SEED):
    pos_users = df.loc[df['label'] == 1, 'customer_id'].unique()   # keep only groups with >=1 positive
    df = df[df['customer_id'].isin(pos_users)]
    pos = df[df['label'] == 1]
    neg = df[df['label'] == 0].sample(frac=1.0, random_state=seed)  # shuffle then cap per user
    neg = neg[neg.groupby('customer_id').cumcount() < neg_per_user]
    return (pd.concat([pos, neg]).sort_values('customer_id').reset_index(drop=True))

In [8]:
# TRAIN frame
train_cands = combine_candidates('train', train_truth['customer_id'].values)
train_df = attach_labels(train_cands, train_truth)
train_df = build_features(train_df, pd.Timestamp(cfg['train_week_start']))
train_df = downsample_negatives(train_df)
save_parquet(train_df, RANKING_DIR / 'ranking_train.parquet')
print('train positives:', int(train_df['label'].sum()), '/', len(train_df),
      f"({train_df['label'].mean():.3%})")

Saved: C:\Users\Michael Fulling\H&M Project\data\processed\ranking\ranking_train.parquet  shape=(1731315, 32)
train positives: 22366 / 1731315 (1.292%)


In [9]:
# VAL frame (no down-sampling: we need every candidate to score the holdout)
val_df = attach_labels(val_cands, val_truth)
val_df = build_features(val_df, pd.Timestamp(cfg['val_week_start']))
val_df = val_df.sort_values('customer_id').reset_index(drop=True)
save_parquet(val_df, RANKING_DIR / 'ranking_val.parquet')
print('val frame rows:', len(val_df), ' positives:', int(val_df['label'].sum()))

Saved: C:\Users\Michael Fulling\H&M Project\data\processed\ranking\ranking_val.parquet  shape=(9280737, 32)
val frame rows: 9280737  positives: 22352


In [10]:
with mlflow.start_run(run_name='05_merge_labels_features'):
    mlflow.log_params({'trend_add': TREND_ADD, 'neg_per_user': NEG_PER_USER})
    mlflow.log_metric('candidate_recall_val', float(rec))
    mlflow.log_metric('avg_candidates_per_val_user', float(avg_cands))
    mlflow.log_metric('train_rows', len(train_df))
    mlflow.log_metric('train_positives', int(train_df['label'].sum()))
print('Logged merge/labels/features run.')

2026/06/11 13:04:35 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



Logged merge/labels/features run.
